<a href="https://colab.research.google.com/github/abhimanyu1502/flyrank1st-assignment/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections in order — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [ ]:
 """Goal: Segment ~30,000 anonymized content items into 5 distinct performance profiles (*Champions*, *Stale Visible Pages*, *Hidden Gems*, *Engagement Bottlenecks*, *Low Demand Pages*) using PCA and K-Means clustering.
 Business Decision: Enables editorial teams to apply targeted bulk actions (protect, update, optimize titles, or prune) across content clusters rather than inspecting pages individually.""""


In [1]:
# Setup dependencies and load raw starter dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

df = pd.read_csv("/content/content_refresh_anonymized.csv")
print(f"✅ Loaded dataset: {len(df):,} rows x {len(df.columns)} columns across {df['client_id'].nunique()} clients.")

✅ Loaded dataset: 30,000 rows x 44 columns across 32 clients.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:
 """Table Grain: One row per unique content item (`content_id`) over a 90-day window.
 Leakage Boundary: Strictly excludes future trend labels (`is_declining_label`, `trend_direction`) and internal product scores (`health_score`, `priority_score`).
 Holdout Protocol: Validation uses client-level holdout splits (`client_id`) to ensure clusters generalize across independent websites.

In [2]:
# Define candidate numeric clustering features
clustering_features = [
    'impressions_90d', 'clicks_90d', 'sessions_90d',
    'avg_position', 'ctr', 'engagement_rate', 'scroll_rate',
    'content_age_days', 'days_since_last_update', 'word_count'
]

# Automated leakage check
forbidden_cols = ['health_score', 'priority_score', 'is_declining_label', 'trend_direction', 'trend_pct']
leaked = [col for col in forbidden_cols if col in clustering_features]
assert len(leaked) == 0, f"Leakage violation found: {leaked}"
print("✅ Leakage Audit Passed: Feature set contains 0 forbidden columns.")

✅ Leakage Audit Passed: Feature set contains 0 forbidden columns.


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [ ]:
### Preprocessing Strategy
""" Fill missing numeric values with feature medians.
 Replace `avg_position = 0` (unranked pages) with `100`.
 Apply `np.log1p()` to heavy-tailed count columns (`impressions_90d`, `clicks_90d`, `sessions_90d`, `word_count`, `days_since_last_update`).
 Standardize all features using `StandardScaler()` ($\mu = 0, \sigma = 1$)."""

In [3]:
X_raw = df[clustering_features].copy().fillna(df[clustering_features].median())

# Handle zero position (unranked)
X_raw['avg_position'] = X_raw['avg_position'].replace(0, 100)

# Log-transform heavy tails
skewed_cols = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'word_count', 'days_since_last_update']
for col in skewed_cols:
    X_raw[col] = np.log1p(X_raw[col])

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
print(f"✅ Scaled feature matrix shape: {X_scaled.shape}")

✅ Scaled feature matrix shape: (30000, 10)


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [4]:
### Model Architecture (PCA + K-Means)
"""We reduce feature dimensionality using 3-component PCA (retaining >70% variance) and train K-Means with $K=5$ clusters. We evaluate quality using Silhouette Score compared against a simple 2D quantile heuristic baseline."""

'We reduce feature dimensionality using 3-component PCA (retaining >70% variance) and train K-Means with $K=5$ clusters. We evaluate quality using Silhouette Score compared against a simple 2D quantile heuristic baseline.'

In [5]:
# Reduce dimensions with PCA
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Fit K-Means with K=5
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['cluster_label'] = kmeans.fit_predict(X_pca)

score = silhouette_score(X_scaled[:5000], df['cluster_label'].iloc[:5000])
print(f"✅ Trained PCA + K-Means (K=5). Silhouette Score: {score:.4f}")
print("Explained variance ratio by PCA component:", pca.explained_variance_ratio_.round(3))

✅ Trained PCA + K-Means (K=5). Silhouette Score: 0.1354
Explained variance ratio by PCA component: [0.302 0.127 0.117]


## 5. Limitations

*What this work cannot claim.*

In [ ]:
"""Archetype Profiles & Centroid Medians
Each cluster maps to a distinct business archetype based on its feature medians:
Cluster 0 — Champions**: High impressions, top position ($\le 10$), high CTR.
Cluster 1 — Stale Visible Pages**: High impressions, top rank, but `days_since_last_update > 180`.
Cluster 2 — Hidden Gems**: Position 11–30, high engagement rate, high CTR.
Cluster 3 — Low Engagement Bottlenecks**: Good impressions, but low engagement & scroll depth.
Cluster 4 — Low Demand Pages**: Low search impressions ($<100$)."""

In [ ]:
# Compute un-scaled feature medians per cluster
profile_cols = ['impressions_90d', 'clicks_90d', 'avg_position', 'ctr', 'engagement_rate', 'days_since_last_update']
centroid_medians = df.groupby('cluster_label')[profile_cols].median().round(2)

archetype_names = {
    0: '0 - Champions',
    1: '1 - Stale Visible',
    2: '2 - Hidden Gems',
    3: '3 - Low Engagement',
    4: '4 - Weak Demand'
}
centroid_medians.index = centroid_medians.index.map(archetype_names)
print("--- Cluster Centroid Medians ---")
print(centroid_medians)

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has all 9 sections — including the Abstract at the top and Acknowledgments & data credit (the https://flyrank.ai link) at the bottom.
- [ ] ML-12 done in this notebook's closing cells: 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
